<a href="https://colab.research.google.com/github/sumitg22/disha-nhs-navigator/blob/main/disha_nhs_navigator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install langchain langchain-anthropic langchain-community faiss-cpu beautifulsoup4 requests -q

In [27]:
from langchain_community.document_loaders import WebBaseLoader

# NHS pages we want to load
urls = [
    "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
    "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
    "https://www.nhs.uk/mental-health/nhs-voluntary-charity-services/nhs-services/",
    "https://www.nhs.uk/nhs-services/students/",
    "https://www.nhs.uk/using-the-nhs/healthcare-abroad/moving-to-england/how-to-access-nhs-services-in-england/"
]

# Load all pages
loader = WebBaseLoader(urls)
documents = loader.load()

print(f"✓ Loaded {len(documents)} NHS pages")
print(f"\nFirst document preview:")
print(documents[0].page_content[:500])

✓ Loaded 5 NHS pages

First document preview:









Register with a GP surgery - NHS









































Skip to main content





NHS







Search the NHS website






Search










                  Health A to Z
                



NHS services




                  Healthy living
                



                  Mental health
                



                  Care and support
                



Browse
              More
            









Home


NHS services


GPs



Back to 
          GPs
     


In [28]:
!pip install langchain-text-splitters -q

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"✓ Total chunks created: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[0].page_content)

✓ Total chunks created: 27

Example chunk:
Register with a GP surgery - NHS









































Skip to main content





NHS







Search the NHS website






Search










                  Health A to Z
                



NHS services




                  Healthy living
                



                  Mental health
                



                  Care and support
                



Browse
              More
            









Home


NHS services


GPs



Back to 
          GPs
        

        Back
      





    
      Register with a GP surgery
    
  
Everyone in England can register with a GP surgery or change their GP surgery for free. You can register with most surgeries online.







Information: 
This page is about registering with a GP surgery or changing your current GP surgery.Find out about:Appointments and bookings at your GP surgeryViewing your test results


In [30]:
!pip install sentence-transformers -q

In [31]:
!pip install langchain-community -q

In [32]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Create embeddings
print("Creating embeddings... this takes 1-2 minutes")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector database
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"✓ Vector database created")
print(f"✓ {len(chunks)} chunks embedded and stored")
print(f"✓ Ready to search")

Creating embeddings... this takes 1-2 minutes


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Vector database created
✓ 27 chunks embedded and stored
✓ Ready to search


In [33]:
!pip install langchain langchain-anthropic langchain-core --upgrade -q

In [49]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Get API key securely
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Connect Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Create prompt template
prompt_template = PromptTemplate.from_template("""You are Disha, an AI assistant
helping South Asians navigate NHS and healthcare services in the UK.
Use the following context from official NHS documents to answer the question.
If you don't know the answer from the context, say "I don't have that information
in my NHS documents — please visit nhs.uk for more details."

Context: {context}

Question: {question}

Answer in a warm, clear, and helpful tone:""")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Create RAG chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print("✓ Disha is ready")
print("✓ Groq connected")
print("✓ NHS knowledge base loaded")
print("✓ RAG chain created")

✓ Disha is ready
✓ Groq connected
✓ NHS knowledge base loaded
✓ RAG chain created


In [45]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.0 MB/s eta 0:00:00


In [50]:
# Test Disha
question = "How do I register with a GP in the UK?"

print(f"Question: {question}")
print(f"\nDisha's answer:")
print("-" * 50)

response = qa_chain.invoke(question)
print(response)

Question: How do I register with a GP in the UK?

Disha's answer:
--------------------------------------------------
Namaste! I'm Disha, your friendly AI assistant. I'm here to help you navigate the NHS and healthcare services in the UK. 

To register with a GP in the UK, you can follow these simple steps:

1. **Find a GP surgery**: You can search online for GP surgeries in your area using the NHS website or by contacting your local integrated care board (ICB) for more information.
2. **Check if the surgery is accepting new patients**: You can call the GP surgery directly to ask if they are accepting new patients. You can also check their website or social media for updates.
3. **Gather the required information**: You'll need to provide basic information to register with a GP surgery, including your name, date of birth, and address. You don't need ID, proof of address, or proof of immigration status.
4. **Register online or in person**: You can register with most GP surgeries online or

In [51]:
# Test 2
question2 = "Am I entitled to free NHS care as an international student?"
print(f"Q: {question2}")
print("-" * 50)
print(qa_chain.invoke(question2))

print("\n")

# Test 3
question3 = "What is NHS 111 and when should I use it?"
print(f"Q: {question3}")
print("-" * 50)
print(qa_chain.invoke(question3))

Q: Am I entitled to free NHS care as an international student?
--------------------------------------------------
Namaste! I'm Disha, your friendly AI assistant. I'm here to help you navigate the NHS and healthcare services in the UK.

Regarding your question, I'd like to clarify that as an international student, you're entitled to free NHS care in the UK, but there are some conditions.

According to the NHS website, you'll need to register with a GP surgery to access NHS services. You can do this without providing ID, proof of address, or proof of immigration status. However, it's essential to note that you might need to provide additional information or documents to help the GP surgery find or transfer your medical records.

As an international student, you might be eligible for a free NHS prescription, but this depends on your individual circumstances. I recommend checking the NHS website or contacting the NHS directly for more information on prescription charges and eligibility.

T